# Colab — Inferencia YOLOv8 sobre test (H6)

Aplica `yolo_best.pt` (de `colab_yolo_train.ipynb`) sobre el split test y exporta predicciones en **formato COCO results** — entrada del framework de evaluación común (H7), comparable directamente con `classical_test.json`.

**GPU recomendada** (~10× más rápido que CPU). *Runtime → Change runtime type → GPU*.

Salida en Drive: `predictions/yolo_test.json`.

## Prerequisitos en Drive

```
MyDrive/grocery-detection/processed/
    train.json, val.json, test.json
    yolo_best.pt                  ← output de colab_yolo_train.ipynb
```

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/arturmoret/GroceryTracker.git"
REPO_DIR = "/content/repo"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

sys.path.insert(0, REPO_DIR + "/scripts")
sys.path.insert(0, REPO_DIR + "/src")
os.chdir(REPO_DIR)

from colab_helper import (
    mount_drive, install_deps, setup_dataset,
    link_processed_to_drive, link_predictions_to_drive, run_script,
)
print("helpers cargados")

In [ ]:
mount_drive()
install_deps()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
print("ultralytics instalado")

In [ ]:
setup_dataset()

In [ ]:
link_processed_to_drive()
link_predictions_to_drive()

In [ ]:
# Regenera labels + split lists (idempotente, segundos).
rc = run_script("scripts/prepare_yolo_dataset.py")
assert rc == 0

## Inferencia + export COCO

Escribe `reports/predictions/yolo_test.json` directo a Drive vía symlink, con write atómico.

In [ ]:
rc = run_script("scripts/run_deep_infer.py")
assert rc == 0

## Listo

`MyDrive/grocery-detection/predictions/yolo_test.json` listo para el framework de evaluación común (H7), comparable contra `classical_test.json`.